# Genre-configurable Stable Audio 3 LoRA workflow

Run on a Colab NVIDIA GPU, in order. Notebook cells execute remotely; the editor's ordinary terminal may still be on your Mac.

This workflow uses prepared 30-second WAV files with same-named text captions. Upload their tar.gz archive to Google Drive first. To prepare another genre, run `prepare_lora_data.py --audio genre_tracks_smoke/hiphop --genre-tag hiphop --out hiphop_lora_smoke --max-chunks 2`, then archive that folder.

A rock adapter is not a hip-hop adapter. Train each genre separately from `small-music-base`; inference uses `small-music`. Existing checkpoints can be loaded for generation without retraining. A 100-step run validates the pipeline, not quality improvement.

## 1. Check GPU and install
Run the next cell once. **Restart the notebook kernel after installation**, then continue at section 2. Do not rerun installation on every reconnect.

In [ ]:
import subprocess
import sys
from pathlib import Path

subprocess.run(['nvidia-smi'], check=True)
repo = Path('/content/MusicLLM')
if not repo.exists():
    subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/rarevalo13/MusicLLM.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'submodule', 'update', '--init', '--recursive'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=torch,torchaudio,torchvision', '-e', str(repo / 'stable-audio-3') + '[lora]', 'torch==2.7.1', 'torchaudio==2.7.1', 'torchvision==0.22.1'], check=True)
print('Restart the kernel now, then continue at section 2. If cloning a private repo fails, configure secure authentication; never embed a token in this notebook.')

## 2. Settings
Edit these values for each experiment. `MODE='train'` creates a new adapter; `MODE='load'` skips data extraction and training and uses `EXISTING_CHECKPOINT`. A notebook uses this settings cell rather than command-line arguments.

Use a new `RUN_NAME` for every training run. Paths below are relative to My Drive. Prompt examples below fit rock/hiphop; supply your own for other genres. After a kernel restart, rerun from this cell.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

GENRE = 'rock'
MODE = 'train'
RUN_NAME = f'{GENRE}_smoke_001'
DATA_ARCHIVE = f'{GENRE}_lora_smoke.tar.gz'
DATA_FOLDER = f'{GENRE}_lora_smoke'
EXISTING_CHECKPOINT = 'MusicLLM_runs/lora_rock_smoke/epoch=4-step=100.ckpt'
TRAIN_STEPS = 100
RANK = 16
BATCH_SIZE = 1
DURATION = 30
SEED = 42
CUSTOM_PROMPTS = []

REPO = Path('/content/MusicLLM')
UPSTREAM = REPO / 'stable-audio-3'
DRIVE = Path('/content/drive/MyDrive')
assert MODE in {'train', 'load'}
assert TRAIN_STEPS > 0 and BATCH_SIZE > 0 and RANK > 0
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME
assert UPSTREAM.is_dir(), 'Run installation first'
presets = {
    'rock': [
        'Instrumental rock, featuring electric guitar, drums, bass, energetic mood, fast tempo',
        'Instrumental rock with indie influences, featuring electric guitar, melancholic mood, medium tempo',
        'Instrumental rock with pop influences, featuring acoustic guitar, drums, happy mood, medium tempo',
    ],
    'hiphop': [
        'Instrumental hip hop, featuring drum machine, bass, piano, relaxed mood, slow tempo',
        'Instrumental hip hop, featuring synthesizer, drum machine, energetic mood, fast tempo',
        'Instrumental hip hop with jazz influences, featuring piano, bass, mellow mood, medium tempo',
    ],
}
PROMPTS = CUSTOM_PROMPTS or presets.get(GENRE, [])
assert PROMPTS, 'Set CUSTOM_PROMPTS for this genre'
print('Mode:', MODE, '| Genre:', GENRE, '| Run:', RUN_NAME)

## 3. Verify dependencies, mount Drive, authenticate
Authorize Drive with the account holding your archive. Authenticate to Hugging Face using the account with model access. Credentials are entered through prompts, never stored in code. Flash Attention warnings are nonfatal for Small.

In [ ]:
import torch
import torchvision
import torchaudio
import pytorch_lightning as pl
from google.colab import drive
from huggingface_hub import notebook_login

print('Torch:', torch.__version__, '| Torchvision:', torchvision.__version__, '| Torchaudio:', torchaudio.__version__)
print('Lightning:', pl.__version__)
assert torch.cuda.is_available(), 'Connect to a GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
if not DRIVE.is_dir():
    drive.mount('/content/drive')
assert DRIVE.is_dir(), 'Drive is not mounted'
notebook_login()
RUN_DIR = DRIVE / 'MusicLLM_runs' / RUN_NAME
COMPARISON_DIR = RUN_DIR / ('comparison' if MODE == 'train' else 'comparison_loaded')
DATA_DIR = REPO / DATA_FOLDER

## 4. Extract and validate training pairs
Skipped in load mode. The archive stays on Drive; extracted audio is stored on the runtime for faster reads. Extraction rejects links, paths outside the destination, and existing destination folders. Reuse an already validated folder by skipping this cell on a rerun.

The archive must contain the folder named by `DATA_FOLDER`. All WAVs must be stereo, 44.1 kHz, 30 seconds for this workflow, with nonempty captions. Your original MP3s are not required.

In [ ]:
if MODE == 'train':
    import tarfile
    import soundfile as sf

    archive = DRIVE / DATA_ARCHIVE
    assert archive.is_file(), f'Upload the archive to Drive: {archive}'
    assert not DATA_DIR.exists(), f'{DATA_DIR} already exists; inspect it and skip this extraction cell if already validated'
    with tarfile.open(archive, 'r:gz') as tar:
        members = [m for m in tar.getmembers() if not any(part.startswith('._') or part == '__MACOSX' for part in Path(m.name).parts)]
        for member in members:
            parts = Path(member.name).parts
            assert parts and parts[0] == DATA_FOLDER, f'Unexpected archive path: {member.name}'
            assert not Path(member.name).is_absolute() and '..' not in parts
            assert member.isfile() or member.isdir(), f'Unsupported archive entry: {member.name}'
        tar.extractall(REPO, members=members, filter='data')
    clips = sorted(p for p in DATA_DIR.glob('*.wav') if not p.name.startswith('._'))
    assert clips, 'No WAV clips found'
    for clip in clips:
        caption = clip.with_suffix('.txt')
        assert caption.is_file() and caption.read_text().strip(), f'Missing/empty caption: {clip}'
        info = sf.info(clip)
        assert info.samplerate == 44100 and info.channels == 2, f'Unexpected audio format: {clip}'
        assert abs(info.duration - DURATION) < 0.1, f'Unexpected clip duration: {clip}'
    print(f'Validated {len(clips)} clips; approximately {TRAIN_STEPS * BATCH_SIZE / len(clips):.1f} dataset passes on one GPU')
    print('Example caption:', clips[0].with_suffix('.txt').read_text())
else:
    print('Load mode: no training data needed')

## 5. Train a new adapter, or select an existing checkpoint
Training is a GPU-consuming action: execute this cell only when ready. The checkpoint interval equals the total steps so even a short run saves a checkpoint. Demos are scheduled beyond the run to keep this smoke test focused; the next section generates comparisons. Steps are optimizer updates, not epochs.

Run settings and source commits are saved alongside outputs on Drive. Existing run directories are rejected to avoid mixing experiments. Load mode does not resume training or alter the adapter.

In [ ]:
if MODE == 'train':
    assert DATA_DIR.is_dir() and any(DATA_DIR.glob('*.wav')), 'Validate training data first'
    assert not RUN_DIR.exists(), f'Choose a fresh RUN_NAME; directory exists: {RUN_DIR}'
    RUN_DIR.mkdir(parents=True)
    command = [sys.executable, 'scripts/train_lora.py', '--model', 'small-music-base', '--data_dir', str(DATA_DIR), '--rank', str(RANK), '--adapter_type', 'dora-rows', '--exclude', 'seconds_total', '--duration', str(DURATION), '--batch_size', str(BATCH_SIZE), '--steps', str(TRAIN_STEPS), '--seed', str(SEED), '--checkpoint_every', str(TRAIN_STEPS), '--demo_every', str(TRAIN_STEPS + 1), '--num_workers', '2', '--save_dir', str(RUN_DIR)]
    settings = {'genre': GENRE, 'steps': TRAIN_STEPS, 'rank': RANK, 'batch_size': BATCH_SIZE, 'duration': DURATION, 'seed': SEED, 'data_archive': DATA_ARCHIVE, 'torch': torch.__version__, 'lightning': pl.__version__, 'gpu': torch.cuda.get_device_name(0), 'command': command, 'repo_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip(), 'upstream_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=UPSTREAM, text=True).strip()}
    (RUN_DIR / 'settings.json').write_text(json.dumps(settings, indent=2))
    subprocess.run(command, cwd=UPSTREAM, check=True)
    checkpoints = sorted(RUN_DIR.rglob('*.ckpt'), key=lambda p: p.stat().st_mtime)
    assert checkpoints, 'No checkpoint saved; inspect training output'
    CHECKPOINT = checkpoints[-1]
else:
    CHECKPOINT = DRIVE / EXISTING_CHECKPOINT
assert CHECKPOINT.is_file(), f'Checkpoint missing: {CHECKPOINT}'
print('Checkpoint:', CHECKPOINT, f'({CHECKPOINT.stat().st_size / 1024**2:.1f} MB)')

## 6. Generate paired base/LoRA samples
Each pair uses the same prompt, seed, duration, and sampling settings. Only adapter strength changes: 0 for base, 1 for LoRA. The base model is downloaded again in a fresh runtime if not cached. Outputs and prompt metadata are saved to Drive. A fresh comparison directory is required to avoid accidental overwrites.

This is a listening smoke test, not a blind evaluation. For final evaluation, reserve tracks/artists before chunking and use a larger fixed prompt set.

In [ ]:
import gc
import soundfile as sf
from stable_audio_3 import StableAudioModel

assert not COMPARISON_DIR.exists(), f'Comparison exists; select a fresh output directory: {COMPARISON_DIR}'
COMPARISON_DIR.mkdir(parents=True)
manifest = {'checkpoint': str(CHECKPOINT), 'genre': GENRE, 'model': 'small-music', 'duration': DURATION, 'steps': 8, 'cfg_scale': 1.0, 'samples': []}
model = StableAudioModel.from_pretrained('small-music')
model.load_lora([str(CHECKPOINT)])
try:
    for i, prompt in enumerate(PROMPTS):
        seed = SEED + i
        for strength, label in [(0.0, 'base'), (1.0, 'lora')]:
            model.set_lora_strength(strength)
            with torch.inference_mode():
                audio = model.generate(prompt=prompt, duration=DURATION, seed=seed, steps=8, cfg_scale=1.0)
            path = COMPARISON_DIR / f'prompt{i}_{label}.wav'
            sf.write(path, audio.detach().cpu().squeeze(0).T.numpy(), 44100)
            manifest['samples'].append({'file': path.name, 'prompt': prompt, 'seed': seed, 'strength': strength})
            (COMPARISON_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2))
            print('Saved:', path)
finally:
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 7. Listen and download
Filenames ending in `_base.wav` have LoRA disabled; `_lora.wav` have it enabled. Download the run folder from Google Drive to your Mac to retain checkpoints, settings, logs, and comparisons. Subtle changes or no improvement are plausible with only 100 steps.

In [ ]:
from IPython.display import Audio, display

for path in sorted(COMPARISON_DIR.glob('*.wav')):
    print(path.name)
    display(Audio(filename=str(path)))
print('Persistent checkpoint:', CHECKPOINT)
print('Persistent comparisons:', COMPARISON_DIR)

## 8. Release the GPU (manual)
Verify the checkpoint and all desired outputs are in Drive first. Local `/content` files are temporary. Closing the editor does not necessarily release the runtime. Execute the following cell only when finished; it terminates the runtime. For future inference, rerun setup and set `MODE='load'` with your Drive checkpoint path.

In [ ]:
assert CHECKPOINT.is_file() and CHECKPOINT.is_relative_to(DRIVE), 'Verify the persistent checkpoint before shutdown'
RELEASE_RUNTIME = False
if RELEASE_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print('Set RELEASE_RUNTIME=True when ready to terminate this runtime.')